# Labelling Strategy

This notebook implements a **weak-supervision labelling strategy** for FIRMS (Fire Information for Resource Management System) hotspot data over India.

The core idea:
- Group satellite detections by approximate location (`group_id` = lat/lon rounded to 3 decimal places)
- Compute per-group persistence and seasonality statistics
- Cross-reference with OpenStreetMap (OSM) industrial/mining features to assign probabilistic labels

## Outputs
- `thermalwatch_labeled.parquet` — labeled training set (used by next notebook)
- `label_summary.json` — labeling audit metadata

## Labels
| Label | Description |
|---|---|
| `industrial_thermal_source` | Persistent, monsoon-active, near industrial OSM feature |
| `mining_thermal_source` | Same as above, but nearest OSM type is `landuse_quarry` |
| `natural_fire` | Seasonal (≤3 months active), not near industry |
| `unknown` | Everything else |



In [ ]:
# Paths — adjust if your data directory differs
FIRMS_DATA_PATH = "../data/processed/firms_india"  # partitioned Parquet dataset
OSM_JSON_PATH = OSM_JSON_PATH  # OSM export (Overpass API)
OUTPUT_PARQUET = "thermalwatch_labeled.parquet"  # labeled training set
OUTPUT_SUMMARY = "label_summary.json"           # labeling audit metadata

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import json

## 2. Load FIRMS Data

Read the partitioned parquet dataset of FIRMS India hotspot detections.

In [32]:
df = pd.read_parquet(FIRMS_DATA_PATH, engine="pyarrow")
df.shape

(5706071, 18)

## 3. Spatial Grouping

Create a `group_id` by rounding lat/lon to 3 decimal places (~100 m resolution).
Each unique `group_id` represents a distinct physical location that has been detected multiple times.

In [33]:
df['group_id'] = df['latitude'].round(3).astype(str) + '_' + df['longitude'].round(3).astype(str)
df['group_id'].nunique()

4714657

## 4. Group-Level Statistics

Aggregate per-group statistics: observation count, first/last detection, mean/std FRP (Fire Radiative Power).

In [34]:
group_stats = df.groupby('group_id').agg(
    obs_count=('observed_at', 'count'),
    first_seen=('observed_at', 'min'),
    last_seen=('observed_at', 'max'),
    mean_frp=('frp', 'mean'),
    std_frp=('frp', 'std'),
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first')
).reset_index()

group_stats.shape
group_stats.sort_values('obs_count', ascending=False).head(10)

,group_id,obs_count,first_seen,last_seen,mean_frp,std_frp,latitude,longitude
2000302,21.921_83.352,278,2022-01-06 20:14:00+00:00,2026-05-11 19:15:00+00:00,4.481619,2.305939,21.92114,83.35157
2000301,21.921_83.351,277,2022-01-18 08:12:00+00:00,2026-05-10 19:34:00+00:00,4.709061,2.337148,21.92088,83.35054
2000761,21.922_83.351,270,2022-01-04 07:35:00+00:00,2026-05-17 08:15:00+00:00,4.825481,2.135231,21.92195,83.35088
2004444,21.92_83.351,259,2022-01-02 07:21:00+00:00,2026-05-25 19:53:00+00:00,4.270077,2.304672,21.92045,83.35114
2004443,21.92_83.35,250,2022-01-02 08:13:00+00:00,2026-05-24 07:43:00+00:00,4.200680,2.172347,21.91957,83.34993
1574749,20.795_85.254,250,2022-01-05 08:04:00+00:00,2026-05-19 20:06:00+00:00,5.445400,3.271496,20.79525,85.25446
2787502,23.684_86.394,246,2022-01-09 07:41:00+00:00,2026-05-30 19:59:00+00:00,4.422154,2.734910,23.68392,86.39390
2004445,21.92_83.352,244,2022-01-01 07:39:00+00:00,2026-05-26 19:34:00+00:00,4.133730,2.288285,21.91964,83.35175
2209826,22.378_87.28,241,2022-01-04 19:11:00+00:00,2026-05-27 19:15:00+00:00,3.572033,1.929907,22.37814,87.27973
2000760,21.922_83.35,237,2022-01-03 19:30:00+00:00,2026-05-28 08:09:00+00:00,4.587468,1.935637,21.92203,83.35016


## 5. Persistence & Seasonality Features

Compute:
- `months_active`: number of distinct calendar months with detections
- `monsoon_active`: whether the group was detected during monsoon (Jun–Sep)
- `frp_cv`: coefficient of variation of FRP (indicates variability)

In [35]:
df['month'] = df['observed_at'].dt.month
df['is_monsoon'] = df['month'].between(6, 9)

persistence = df.groupby('group_id').agg(
    months_active=('month', lambda x: x.nunique()),
    monsoon_obs_count=('is_monsoon', 'sum'),
).reset_index()

group_stats = group_stats.merge(persistence, on='group_id')

group_stats['monsoon_active'] = group_stats['monsoon_obs_count'] > 0
group_stats['frp_cv'] = group_stats['std_frp'] / group_stats['mean_frp']

group_stats.sort_values('obs_count', ascending=False).head(10)[
    ['group_id', 'obs_count', 'months_active', 'monsoon_active', 'frp_cv']
]

,group_id,obs_count,months_active,monsoon_active,frp_cv
2000302,21.921_83.352,278,12,True,0.514533
2000301,21.921_83.351,277,12,True,0.496309
2000761,21.922_83.351,270,12,True,0.442491
2004444,21.92_83.351,259,12,True,0.539726
2004443,21.92_83.35,250,12,True,0.517142
1574749,20.795_85.254,250,12,True,0.600782
2787502,23.684_86.394,246,12,True,0.618456
2004445,21.92_83.352,244,12,True,0.553564
2209826,22.378_87.28,241,12,True,0.540283
2000760,21.922_83.35,237,12,True,0.421940


In [36]:
group_stats['months_active'].value_counts().sort_index()

months_active
1     4573837
2      108285
3        7667
4        4561
5        3601
6        3096
7        2606
8        2305
9        2085
10       2048
11       2100
12       2466
Name: count, dtype: int64

In [37]:
group_stats['monsoon_active'].value_counts()

monsoon_active
False    4576753
True      137904
Name: count, dtype: int64

In [38]:
group_stats['obs_count'].describe()

count    4.714657e+06
mean     1.210283e+00
std      3.143841e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      2.780000e+02
Name: obs_count, dtype: float64

### Persistent groups
How many groups are active ≥9 months AND monsoon-active? These are strong industrial candidates.

In [39]:
((group_stats['months_active'] >= 9) & (group_stats['monsoon_active'])).sum()

8699

## 6. OSM Industrial Feature Cross-Reference

Load the OpenStreetMap industrial features (exported via Overpass API) and build a KD-tree for nearest-neighbor lookup.

**OSM tags used:**
- `power=*` (power plants, substations)
- `landuse=industrial` / `landuse=quarry`
- `man_made=chimney` / `man_made=works`
- `industrial=*`

In [ ]:
with open(OSM_JSON_PATH) as f:
    osm_raw = json.load(f)

len(osm_raw['elements'])

In [41]:
osm_features = []

for el in osm_raw['elements']:
    tags = el.get('tags', {})
    
    if el['type'] == 'node':
        lat, lon = el.get('lat'), el.get('lon')
    else:  # way — use center
        center = el.get('center', {})
        lat, lon = center.get('lat'), center.get('lon')
    
    if lat is None or lon is None:
        continue
    
    # determine primary tag type
    if 'power' in tags:
        feature_type = f"power_{tags['power']}"
    elif 'landuse' in tags:
        feature_type = f"landuse_{tags['landuse']}"
    elif 'man_made' in tags:
        feature_type = f"man_made_{tags['man_made']}"
    elif 'industrial' in tags:
        feature_type = f"industrial_{tags['industrial']}"
    else:
        feature_type = "other"
    
    osm_features.append({
        'osm_id': el['id'],
        'lat': lat,
        'lon': lon,
        'feature_type': feature_type,
        'name': tags.get('name', 'Unnamed')
    })

osm_df = pd.DataFrame(osm_features)
osm_df.shape
osm_df['feature_type'].value_counts()

feature_type
landuse_industrial               35729
landuse_quarry                   12639
man_made_chimney                 12227
power_plant                      10100
man_made_works                    8982
                                 ...  
industrial_Research institute        1
industrial_ice_factory               1
man_made_grinding_mill               1
industrial_mineral_processing        1
power_utility                        1
Name: count, Length: 95, dtype: int64

### KD-Tree nearest-neighbor lookup

For each FIRMS group, find the nearest OSM industrial feature (by Euclidean distance in degrees).

In [42]:
from scipy.spatial import cKDTree

# Build KD-tree from OSM feature coordinates
osm_coords = osm_df[['lat', 'lon']].values
tree = cKDTree(osm_coords)

# Query nearest OSM feature for every group
group_coords = group_stats[['latitude', 'longitude']].values
distances, indices = tree.query(group_coords, k=1)

group_stats['nearest_osm_distance_deg'] = distances
group_stats['nearest_osm_type'] = osm_df.iloc[indices]['feature_type'].values
group_stats['nearest_osm_name'] = osm_df.iloc[indices]['name'].values

group_stats[['group_id', 'nearest_osm_distance_deg', 'nearest_osm_type']].head(10)

,group_id,nearest_osm_distance_deg,nearest_osm_type
0,10.001_76.892,0.041855,power_plant
1,10.001_76.893,0.042788,power_plant
2,10.001_77.164,0.027336,landuse_quarry
3,10.001_77.186,0.023330,landuse_quarry
4,10.001_77.229,0.012910,landuse_industrial
5,10.001_77.501,0.009505,landuse_industrial
6,10.001_77.573,0.003397,landuse_industrial
7,10.001_77.575,0.003624,landuse_industrial
8,10.001_77.797,0.067233,landuse_quarry
9,10.001_78.021,0.043940,landuse_industrial


In [43]:
group_stats['nearest_osm_distance_km'] = group_stats['nearest_osm_distance_deg'] * 111
group_stats['near_industrial'] = group_stats['nearest_osm_distance_km'] <= 2.0  # 2km threshold

group_stats['near_industrial'].value_counts()

near_industrial
False    4403978
True      310679
Name: count, dtype: int64

### OSM corroboration rate

Of the persistent (≥9 months active + monsoon active) groups, what fraction are near industrial OSM features?
A high rate validates the labeling approach.

In [44]:
persistent_mask = (group_stats['months_active'] >= 9) & (group_stats['monsoon_active'])
group_stats[persistent_mask]['near_industrial'].value_counts()

near_industrial
True     7812
False     887
Name: count, dtype: int64

In [45]:
group_stats[persistent_mask]['near_industrial'].value_counts(normalize=True)

near_industrial
True     0.898034
False    0.101966
Name: proportion, dtype: float64

## 7. Label Assignment

Apply rule-based weak-supervision labels using three conditions:

| Priority | Condition | Label | Confidence |
|---|---|---|---|
| 1 | `months_active ≥ 9` AND `monsoon_active` AND `near_industrial` | `industrial_thermal_source` | 0.90 |
| 2 | `months_active ≥ 9` AND `monsoon_active` AND NOT `near_industrial` | `unknown` | 0.40 |
| 3 | `months_active ≤ 3` AND NOT `near_industrial` | `seasonal_fire_candidate` | 0.50 |
| default | everything else | `unknown` | 0.30 |

In [48]:
is_persistent = (group_stats['months_active'] >= 9) & (group_stats['monsoon_active'])
is_seasonal = group_stats['months_active'] <= 3
near_industrial = group_stats['near_industrial']

conditions = [
    is_persistent & near_industrial,
    is_persistent & ~near_industrial,
    is_seasonal & ~near_industrial,
]

labels = ['industrial_thermal_source', 'unknown', 'seasonal_fire_candidate']
confidences = [0.9, 0.4, 0.5]
reasons = ['persistent_and_near_industrial', 'persistent_but_no_industrial_corroboration', 'seasonal_no_industrial_context']

group_stats['label'] = np.select(conditions, labels, default='unknown')
group_stats['label_confidence'] = np.select(conditions, confidences, default=0.3)
group_stats['label_reason'] = np.select(conditions, reasons, default='ambiguous_pattern')

group_stats['label'].value_counts()

label
seasonal_fire_candidate      4400149
unknown                       306696
industrial_thermal_source       7812
Name: count, dtype: int64

### Refine: split industrial → mining

If the nearest OSM feature type is `landuse_quarry`, re-label as `mining_thermal_source`.

In [49]:
def refine_industrial_label(row):
    if row['label'] == 'industrial_thermal_source':
        if row['nearest_osm_type'] == 'landuse_quarry':
            return 'mining_thermal_source'
        else:
            return 'industrial_thermal_source'
    return row['label']

group_stats['label'] = group_stats.apply(refine_industrial_label, axis=1)
group_stats['label'].value_counts()

label
seasonal_fire_candidate      4400149
unknown                       306696
industrial_thermal_source       4936
mining_thermal_source           2876
Name: count, dtype: int64

### Rename seasonal candidate → natural_fire

In [50]:
group_stats['label'] = group_stats['label'].replace('seasonal_fire_candidate', 'natural_fire')
group_stats['label'].value_counts()

label
natural_fire                 4400149
unknown                       306696
industrial_thermal_source       4936
mining_thermal_source           2876
Name: count, dtype: int64

### Label quality check: obs_count distributions per class

In [51]:
group_stats.groupby('label')['obs_count'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
industrial_thermal_source,4936.0,58.124392,37.968151,10.0,30.0,47.0,77.0,278.0
mining_thermal_source,2876.0,64.445758,41.678882,9.0,33.0,53.0,84.0,246.0
natural_fire,4400149.0,1.056527,0.254149,1.0,1.0,1.0,1.0,15.0
unknown,306696.0,1.907247,4.590773,1.0,1.0,1.0,1.0,173.0


## 8. Training Set Construction

Keep all industrial + mining samples (rare minority classes). 
Downsample `natural_fire` **stratified by first-detection month** to preserve seasonal distribution.
Downsample `unknown` to a smaller budget (least reliable class).

In [54]:
natural_all = group_stats[group_stats['label'] == 'natural_fire']
natural_sample = natural_all.groupby(natural_all['first_seen'].dt.month, group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
)
natural_sample['label'].value_counts()
len(natural_sample)

label
natural_fire                 24000
unknown                      10000
industrial_thermal_source     4936
mining_thermal_source         2876
Name: count, dtype: int64

In [55]:
industrial = group_stats[group_stats['label'] == 'industrial_thermal_source']
mining = group_stats[group_stats['label'] == 'mining_thermal_source']

natural_all = group_stats[group_stats['label'] == 'natural_fire']
natural_sample = natural_all.groupby(natural_all['first_seen'].dt.month, group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
)

unknown_sample = group_stats[group_stats['label'] == 'unknown'].sample(n=10000, random_state=42)

training_set = pd.concat([industrial, mining, natural_sample, unknown_sample], ignore_index=True)
training_set['label'].value_counts()

{'total_groups_in_full_dataset': 4714657,
 'total_training_groups': 41812,
 'class_distribution': {'natural_fire': 24000,
  'unknown': 10000,
  'industrial_thermal_source': 4936,
  'mining_thermal_source': 2876},
 'osm_corroboration_rate': 0.8980342568111277,
 'labeling_method': 'weak_supervision: FIRMS persistence + OSM industrial proximity',
 'osm_features_used': 80687,
 'notes': 'natural_fire stratified by first-detection month to preserve seasonal distribution; industrial/mining classes use full available population due to small size'}

## 9. Save Outputs

In [ ]:
training_set.to_parquet(OUTPUT_PARQUET, engine="pyarrow", index=False)
print(f"Saved {len(training_set)} rows to {"thermalwatch_labeled.parquet"}")

In [ ]:
label_summary = {
    "total_groups_in_full_dataset": int(len(group_stats)),
    "total_training_groups": int(len(training_set)),
    "class_distribution": training_set['label'].value_counts().to_dict(),
    "osm_corroboration_rate": float((group_stats[persistent_mask]['near_industrial']).mean()),
    "labeling_method": "weak_supervision: FIRMS persistence + OSM industrial proximity",
    "osm_features_used": int(len(osm_df)),
    "notes": "natural_fire stratified by first-detection month to preserve seasonal distribution; industrial/mining classes use full available population due to small size"
}

with open(OUTPUT_SUMMARY, "w") as f:
    json.dump(label_summary, f, indent=2)

label_summary